# Lab 4: Markov Chain and Q-Learning

## Markov Chain
* **Shift from one state to another**
* **Depend on next step not sequence of past step.**
* **Memoryless**

### Properties
1. **State space**
2. **Transition Probability Matrix**
3. **Stationary distribution**
   * No change probability distribution.

## Markov Decision Process (MDP)
>> Sequential Decision making problem where outcome of an action is uncertain.

>> Components: State (S), Action (A), transition (P), Reward function (R), $\gamma$ (discount Rate)

## Q-Learning
* Trial and Error algorithm
* Learn the long term value
* Specific action in specific situation
* **Q** $\rightarrow$ Quality
* Lookup table (Q-table)
* *If state "S" and take action "A", How good will my total future pay off?*

### Mathematical Equation
$$Q(s, a) \leftarrow Q(s, a) + \alpha \left[ R + \gamma \max_{a'} Q(s', a') - Q(s, a) \right]$$

Where:
* $Q(s,a)$ (old value) $\rightarrow$ agent current think action a is worth
* $\alpha$ (learning rate) $\rightarrow$ "0" and "1" [0: nothing learn, 1: Learn new]
* $R$ (immediate Reward) $\rightarrow$ penalty score
* $\gamma$ (Discount factor) $\rightarrow$ "0" and "1" [reward against immediate one]
* $s'$ $\rightarrow$ Next state
* $\max_{a'} Q(s', a')$ $\rightarrow$ Best Expected future value

## Example: 4x4 Grid World

**Grid:**
| S0 | S1 | S2 | S3 |
|---|---|---|---|
| S4 | **S5** | S6 | S7 |
| S8 | S9 | **S10**| S11|
| S12| S13| S14| **S15**|

* **Start State:** S0
* **Goal State:** S15
* **Obstacles:** S5 & S10
* **Parameters:** $\alpha = 0.5$, $\gamma = 0.9$
* **Rewards:** 
  * $R = +10$ (Goal)
  * $R = -10$ (obstacle)
  * $R = -1$ (Normal)
* **Actions:** $\uparrow$ (U), $\downarrow$ (D), $\leftarrow$ (L), $\rightarrow$ (R)

### Initial Step:
**Q-table:** Intialized with zeros for all states ($S$) and actions.

### Episode Walkthrough

**Ep 1:**
* $S0 \xrightarrow{R} S1$
  * $Q(S0, R) = 0 + 0.5[-1 + 0.9(0) - 0] = -0.5$
* $S1 \xrightarrow{D} S5$ *(obstacle)*
  * $Q(S1, D) = 0 + 0.5[-10 + 0.9(0) - 0] = -5$

**Ep 2:**
* $S0 \xrightarrow{D} S4$
  * $Q(S0, D) = -0.5$
* ... *(skipping intermediate steps)* ...
* $S14 \xrightarrow{R} S15$ *(goal)*
  * $Q(S14, R) = 0 + 0.5[10 + 0.9(0) - 0] = 5$

**Ep 3 (Back propagate):**
* $Q(S14, R) = 5$
* $S13 \xrightarrow{R} S14$
  * Assume $Q(S13, R)$ was previously $-0.5$
  * $Q(S13, R) \leftarrow -0.5 + 0.5[-1 + 0.9(5) - (-0.5)]$
  * $= -0.5 + 0.5[-1 + 4.5 + 0.5] = -0.5 + 0.5[4] = -0.5 + 2 = 1.5$

## Code Implementation
Below is the Python code simulating the Q-Learning process on this 4x4 Grid Environment.

In [1]:
import numpy as np
import random

# Environment settings
grid_size = 4
num_states = grid_size * grid_size
num_actions = 4 # 0: Up, 1: Down, 2: Left, 3: Right
actions = ['U', 'D', 'L', 'R']

# Rewards and special states
start_state = 0
goal_state = 15
obstacles = [5, 10]

R_normal = -1
R_goal = 10
R_obstacle = -10

# Hyperparameters
alpha = 0.5
gamma = 0.9
epsilon = 0.1 # Exploration rate
episodes = 1000

# Initialize Q-table
Q = np.zeros((num_states, num_actions))

def get_next_state(state, action):
    row = state // grid_size
    col = state % grid_size
    
    if action == 0 and row > 0:       # Up
        next_state = state - grid_size
    elif action == 1 and row < grid_size - 1: # Down
        next_state = state + grid_size
    elif action == 2 and col > 0:       # Left
        next_state = state - 1
    elif action == 3 and col < grid_size - 1: # Right
        next_state = state + 1
    else:
        next_state = state # Hit wall, stay in same state
        
    return next_state

# Training the agent
for episode in range(episodes):
    state = start_state
    done = False
    
    while not done:
        # Epsilon-greedy action selection
        if random.uniform(0, 1) < epsilon:
            action = random.randint(0, 3)
        else:
            action = np.argmax(Q[state, :])
            
        next_state = get_next_state(state, action)
        
        # Check rewards and termination
        if next_state == goal_state:
            reward = R_goal
            done = True
        elif next_state in obstacles:
            reward = R_obstacle
            done = True
        else:
            reward = R_normal
            done = False
            
        # Update Q-Value
        max_next_q = np.max(Q[next_state, :]) if not done else 0
        Q[state, action] = Q[state, action] + alpha * (reward + gamma * max_next_q - Q[state, action])
        
        state = next_state

print("Training completed.")
print("\nFinal Q-Table (Rounded):")
print(np.round(Q, 2))

# Extracting the optimal path
optimal_path = [start_state]
state = start_state
while state != goal_state and state not in obstacles:
    action = np.argmax(Q[state, :])
    next_state = get_next_state(state, action)
    if next_state == state: # Prevents infinite loop if stuck
        break
    optimal_path.append(next_state)
    state = next_state

print("\nOptimal Path from S0 to Goal (S15):")
print(" -> ".join([f"S{s}" for s in optimal_path]))


Training completed.

Final Q-Table (Rounded):
[[  0.63   1.81   0.63   1.81]
 [  1.81 -10.     0.63   3.12]
 [  3.12   4.58   1.81   4.58]
 [  1.58   6.2    0.65  -1.43]
 [ -2.93   3.12  -2.65  -9.38]
 [  0.     0.     0.     0.  ]
 [  3.12 -10.   -10.     6.2 ]
 [  4.58   8.     4.58   6.2 ]
 [ -0.1    4.58  -2.47  -2.08]
 [ -5.     0.37  -1.45  -7.5 ]
 [  0.     0.     0.     0.  ]
 [  6.2   10.   -10.     8.  ]
 [  1.69   1.57  -1.43   6.2 ]
 [ -1.09  -0.98  -1.3    8.  ]
 [ -5.     5.81   4.44  10.  ]
 [  0.     0.     0.     0.  ]]

Optimal Path from S0 to Goal (S15):
S0 -> S1 -> S2 -> S6 -> S7 -> S11 -> S15


## Conclusion and Discussions

**Conclusion:**
In this lab, we successfully explored the concepts of Markov Chains, Markov Decision Processes (MDPs), and the Q-Learning algorithm. Through a practical 4x4 grid example, we manually traced the trial-and-error approach, examining how an agent updates its Q-table using the Bellman equation. By interacting with the environment, receiving penalties for obstacles (-10) and normal moves (-1), and securing a reward (+10) upon reaching the goal, the agent dynamically learns the optimal path from the start state to the goal.

**Discussions:**
1. **Exploration vs. Exploitation:** The trial-and-error nature of Q-Learning highlights the balance required between exploring new paths and exploiting known high-reward paths. In our programmatic implementation, we employ an $\epsilon$-greedy strategy to handle this.
2. **Impact of Hyperparameters:**
    - **Learning Rate ($\alpha=0.5$):** Determines how much new information overrides old information. A value of 0.5 allows the agent to steadily integrate recent experiences.
    - **Discount Factor ($\gamma=0.9$):** A value close to 1 indicates that the agent cares significantly about long-term rewards, which is critical in a grid maze where the immediate reward is usually negative until the final goal is reached.
3. **Memoryless Property:** As part of the Markov property, the agent evaluates the next action strictly based on the current state (like S13) without needing the history of how it arrived there.
4. **Value Backpropagation:** Our manual trace (Ep 3) clearly demonstrated how the high reward received at the goal state (S15) backpropagates to previous states (S14 $\rightarrow$ S13). This allows the entire Q-table to eventually reflect the true "quality" of actions from any state in the grid.